# 🚀 CLIR-SkateFormer: All-in-One Framework
## Concept Logic Iterative Refinement for Skeleton Action Recognition

> **Mục tiêu**: Xây dựng mô hình nhận diện hành động từ dữ liệu khung xương (Skeleton Action Recognition) có khả năng **suy luận logic minh bạch, giải thích được (Explainable AI)** và đạt độ chính xác cao trên benchmark **NTU RGB+D (NTU60 / NTU120)**.
>
> **Kiến trúc cốt lõi**:
> - **Perception Backbone**: SkateFormer trích xuất biểu diễn không gian - thời gian theo từng phân vùng khớp cơ thể.
> - **Logic Reasoner**: CRL (Concept Reasoner with Logic) mã hóa tri thức hành động bằng luật DNF phân cấp (L1 AND $\rightarrow$ L2a AND2 $\parallel$ L2b OR1 $\rightarrow$ ActionHead).
> - **Iterative Refinement**: Tinh chỉnh khái niệm vòng lặp ($K=2$) dẫn dắt bởi Logic Evidence ($r^t \in \mathbb{R}^{512}$).
> - **Pure STE Annealing**: Huấn luyện thẳng với Straight-Through Estimator và Temperature Annealing ($2.0 \rightarrow 10.0$).

---

### 🗺️ Lộ trình Toàn diện (All-in-One Roadmap)
| Giai đoạn | Nội dung cốt lõi | Trạng thái |
| :--- | :--- | :--- |
| **Phần I: Dataset & Feeder** | 72 Concept Ontology, Augmentation Tools, NTU 4-Modality Feeder, Interaction Feeder, DataLoaders, Visualizer | 🟢 **Đã hoàn thiện** |
| **Phần II: Perception Backbone** | SkateFormer Backbone (Part-grouped Attention, Partition 24 Joints) | ⏳ Sẵn sàng tích hợp |
| **Phần III: CRL Logic Layers** | LiteralLayer (BinarizeSTE), ConjunctionLayer, DisjunctionLayer, CRLReasoner | ⏳ Sẵn sàng tích hợp |
| **Phần IV: Training & Validation** | Pure STE Temperature Annealing, Multi-task Loss (CE + Asymmetric BCE + Logic Reg) | ⏳ Sẵn sàng tích hợp |
| **Phần V: Rule Extraction & XAI** | Trích xuất luật DNF tường minh, đánh giá Fidelity, Precision, Coverage | ⏳ Sẵn sàng tích hợp |

---

### 📊 Cấu trúc Dữ liệu Bộ xương NTU RGB+D:
- **Tensor Shape**: `(N, C, T, V, M)`
  - $N$: Batch size
  - $C = 3$: Tọa độ 3 chiều $(x, y, z)$
  - $T = 64$: Số lượng khung hình chuẩn hóa (Temporal window)
  - $V = 25$ (hoặc $24$ khớp khi phân vùng loại bỏ cột sống đáy #21)
  - $M = 2$: Tối đa 2 người trong mỗi mẫu
- **4 Phương thức dữ liệu (Modalities)**:
  1. **Joint (`j`)**: Tọa độ khớp nguyên bản
  2. **Bone (`b`)**: Vector xương nối giữa các cặp khớp theo cây động học
  3. **Joint-Motion (`jm`)**: Vận tốc tức thời của khớp $\Delta x_t = x_{t+1} - x_t$
  4. **Bone-Motion (`bm`)**: Vận tốc thay đổi của vector xương
- **72 Concept Ontology**:
  - $52$ khái niệm không gian (đầu, thân, tay, chân)
  - $20$ khái niệm thời gian (hướng chuyển động, nhịp độ, thứ tự hành vi)


# 📦 PHẦN I: HỆ THỐNG DỮ LIỆU & BỘ NẠP (DATASET & FEEDER PIPELINE)
Các block dưới đây xây dựng toàn bộ nền tảng tiền xử lý, tăng cường dữ liệu, nạp dữ liệu và kiểm thử cho toàn bộ hệ thống CLIR.


## 🛠️ Block 1: Môi trường, Thư viện & Thiết lập Cố định Seed
Block này nạp các thư viện toán học, học sâu, và xử lý dữ liệu chuẩn (`torch`, `numpy`, `matplotlib`). Cố định Seed đảm bảo tính tái lập (reproducibility) trong toàn bộ quá trình tăng cường dữ liệu.


In [ ]:
# Cài đặt các gói cần thiết nếu chạy trên Colab / Môi trường mới
# !pip install torch torchvision numpy matplotlib tqdm -q

import os
import sys
import math
import random
import csv
import json
import types
from pathlib import Path

# Đảm bảo mã hóa UTF-8 an toàn cho console Windows / Colab
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# 1. Cố định Seed cho tính tái lập
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# 2. Cấu hình Thiết bị (GPU / CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"[Python] Version: {sys.version.split()[0]}")
print(f"[PyTorch] Version: {torch.__version__}")
print(f"[Device] Using: {DEVICE}")
if torch.cuda.is_available():
    print(f"[GPU] Device Name: {torch.cuda.get_device_name(0)}")


## 🧬 Block 2: Concept Ontology Engine (72 Khái niệm Ngữ nghĩa)
CLIR liên kết mỗi lớp hành động (Action Class $A_1 \dots A_{60}$ hoặc $A_{120}$) với **72 khái niệm hành vi** (Concept targets $c \in \{0, 1\}^{72}$):
- **52 Khái niệm Không gian (Spatial Concepts)**: Chi tiết chuyển động ở các vùng giải phẫu (`head_nod`, `hand_grasp`, `arm_raise`, `hip_bend`, `leg_kick`, ...).
- **20 Khái niệm Thời gian (Temporal Concepts)**: Chi tiết động học và trình tự (`temporal_direction_motion_upward`, `temporal_dynamics_speed_fast`, `temporal_sequence_seq_hands_first`, ...).

> [!NOTE]
> Bảng dữ liệu chuẩn ban đầu có 74 cột. Chúng tôi tự động loại bỏ 2 khái niệm rỗng (`foot_tap` tại index 50 và `temporal_sequence_seq_cascading` tại index 65) để đạt chính xác **72 concepts** chuẩn hóa của benchmark REASON / CLIR.


In [ ]:
# Tạo file ontology CSV nếu chưa tồn tại
CONCEPT_CSV_CONTENT = """﻿action_class,head_nod,head_shake,head_tilt_down,head_tilt_up,head_rotate,head_static,hand_grasp,hand_release,hand_wave,hand_point,hand_clap,hand_manipulate,hand_fist,hand_open,hand_touch,hand_lift_to_face,hand_circular,arm_raise,arm_lower,arm_extend,arm_retract,arm_circular,arm_swing,arm_cross,arm_hold,arm_alternating,arm_overhead,arm_lateral,hip_bend,hip_extend,hip_sit,hip_squat,hip_rotate,hip_static_stand,hip_static_sit,hip_sway,leg_step,leg_kick,leg_jump,leg_squat,leg_lift,leg_static_stand,leg_static_sit,leg_alternating,leg_balance,foot_step,foot_kick,foot_jump,foot_pivot,foot_static,foot_slide,interaction_two_person,temporal_direction_motion_upward,temporal_direction_motion_downward,temporal_direction_motion_forward,temporal_direction_motion_backward,temporal_direction_motion_converging,temporal_direction_motion_diverging,temporal_direction_motion_reversible,temporal_sequence_seq_hands_first,temporal_sequence_seq_legs_first,temporal_sequence_seq_body_first,temporal_sequence_seq_simultaneous,temporal_sequence_seq_alternating,temporal_dynamics_speed_slow,temporal_dynamics_speed_fast,temporal_dynamics_speed_accelerating,temporal_dynamics_speed_decelerating,temporal_dynamics_rhythm_regular,temporal_dynamics_rhythm_irregular,temporal_dynamics_duration_brief,temporal_dynamics_duration_sustained,hand_to_hand_interaction,one_hand_manipulates_other_hand
drink_water,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0
eat_meal,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0
brush_teeth,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0
brush_hair,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0
drop,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0
pick_up,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0
throw,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0
sit_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0
stand_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0
clapping,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0
reading,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0
writing,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1,0,0
tear_up_paper,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0
put_on_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
take_off_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
put_on_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0
take_off_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
put_on_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
take_off_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
put_on_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
take_off_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
cheer_up,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
hand_waving,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0
kicking_something,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0
reach_into_pocket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
hopping,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0
jump_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
phone_call,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
play_with_phone_tablet,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0
type_on_a_keyboard,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,0
point_to_something,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
taking_a_selfie,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
check_time_(from_watch),0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
rub_two_hands,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,0
nod_head_bow,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0
shake_head,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0
wipe_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0
salute,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
put_palms_together,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0
cross_hands_in_front,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
sneeze_cough,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0
staggering,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0
falling_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0
headache,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
chest_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
back_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
neck_pain,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0
nausea_vomiting,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0
fan_self,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0
punch_slap,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0
kicking,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0
pushing,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0
pat_on_back,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0
point_finger,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
hugging,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
giving_object,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0
touch_pocket,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
shaking_hands,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,1,0
walking_towards,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0
walking_apart,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0
put_on_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
take_off_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
shoot_at_basket,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
bounce_ball,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0
tennis_bat_swing,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0
juggle_table_tennis_ball,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1,0,0
hush,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
flick_hair,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
thumb_up,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
thumb_down,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
make_OK_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
make_victory_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
staple_book,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
counting_money,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0
cutting_nails,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1,1,1
cutting_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0
snap_fingers,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
open_bottle,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0
sniff_smell,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0
squat_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0
toss_a_coin,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
fold_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0
ball_up_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0
play_magic_cube,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,0
apply_cream_on_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0
apply_cream_on_hand,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,1,1
put_on_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
take_off_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
put_object_into_bag,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
take_object_out_of_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
open_a_box,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0
move_heavy_objects,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0
shake_fist,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,0
throw_up_cap_hat,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
capitulate,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
cross_arms,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
arm_circles,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0
arm_swings,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0
run_on_the_spot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1,0,0
butt_kicks,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1,0,0
cross_toe_touch,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0
side_kick,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0
yawn,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0
stretch_oneself,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0
blow_nose,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
hit_with_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0
wield_knife,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
knock_over,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
grab_stuff,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0
shoot_with_gun,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
step_on_foot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0
high-five,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,1,0
cheers_and_drink,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0
carry_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0
take_a_photo,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
follow,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0
whisper,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
exchange_things,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0
support_somebody,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
rock-paper-scissors,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,1,0"""

os.makedirs("concepts", exist_ok=True)
concept_file_path = Path("concepts/ntu_rgbd_60_120.csv")
if not concept_file_path.exists():
    concept_file_path.write_text(CONCEPT_CSV_CONTENT, encoding="utf-8")
    print(f"[Init] Da khoi tao file ontology: {concept_file_path}")
else:
    print(f"[Init] File ontology da co san tai: {concept_file_path}")

class ConceptOntology:
    """Quản lý và tra cứu Concept Ontology cho NTU RGB+D 60/120."""
    def __init__(self, csv_path="concepts/ntu_rgbd_60_120.csv", remove_null_concepts=True):
        self.csv_path = Path(csv_path)
        self.remove_null_concepts = remove_null_concepts
        self.action_names = []
        self.concept_names = []
        self.concept_matrix = None  # shape: (num_actions, 72)
        self.load_ontology()

    def load_ontology(self):
        with open(self.csv_path, 'r', newline='', encoding='utf-8-sig') as f:
            reader = list(csv.DictReader(f))
        
        if not reader or 'action_class' not in reader[0]:
            raise ValueError(f"File CSV ontology khong hop le: {self.csv_path}")
            
        all_cols = [k for k in reader[0].keys() if k != 'action_class']
        
        # Lọc bỏ 2 null concepts nếu có
        if self.remove_null_concepts and len(all_cols) == 74:
            valid_cols = [c for i, c in enumerate(all_cols) if i not in (50, 65)]
        elif self.remove_null_concepts and len(all_cols) > 72:
            valid_cols = [c for c in all_cols if c not in ('foot_tap', 'temporal_sequence_seq_cascading')]
        else:
            valid_cols = all_cols
            
        if len(valid_cols) != 72:
            raise ValueError(f"Ky vong 72 concepts, nhan duoc {len(valid_cols)} concepts!")
            
        self.concept_names = valid_cols
        matrix = []
        for row in reader:
            self.action_names.append(row['action_class'])
            matrix.append([float(row[col]) for col in valid_cols])
            
        self.concept_matrix = np.array(matrix, dtype=np.float32)
        print(f"[Ontology] Da nap thanh cong: {len(self.action_names)} hanh dong x {self.concept_matrix.shape[1]} khai niem.")

    def get_concept_vector(self, action_idx):
        """Lấy nhãn vector concept 72 chiều cho action_idx."""
        return self.concept_matrix[action_idx]

    def get_active_concepts(self, action_idx):
        """Trả về danh sách tên các concepts được gán giá trị 1 cho hành động."""
        vec = self.concept_matrix[action_idx]
        return [self.concept_names[i] for i, val in enumerate(vec) if val > 0.5]

# Khởi tạo và kiểm tra nhanh Ontology
ontology = ConceptOntology()
sample_actions = [0, 9, 22]  # A1: drink_water, A10: clapping, A23: cheer_up
print("")
print("[Ontology Sample Check]:")
for act_id in sample_actions:
    act_name = ontology.action_names[act_id]
    active_c = ontology.get_active_concepts(act_id)
    print(f"- [Action {act_id+1:02d}: {act_name}] ({len(active_c)} active concepts): {', '.join(active_c[:4])} ...")


## 🦾 Block 3: Công cụ Tiền xử lý & Tăng cường Khung xương (`tools.py`)
Block này cung cấp toàn bộ các phép biến đổi toán học và động học trên tensor khung xương `(C, T, V, M)`:
1. **Cắt khung hình & Chuẩn hóa độ dài (Temporal Cropping)**:
   - `valid_crop_uniform`: Lấy mẫu đồng đều theo phân vị thời gian.
   - `valid_crop_resize`: Cắt khung hình hợp lệ và nội suy tuyến tính về kích thước cửa sổ chuẩn ($T=64$).
2. **Biến đổi Hình học & Nhiễu Không gian (Geometric Augmentations)**:
   - `rotate`: Xoay ngẫu nhiên quanh trục X/Y/Z một góc $\theta \in [-30^{\circ}, +30^{\circ}]$.
   - `shear`: Biến dạng cắt ngẫu nhiên (shear matrix).
   - `scale`: Co giãn kích thước khung xương ngẫu nhiên.
   - `spatial_flip`: Đổi xứng khớp trái/phải theo giải phẫu cơ thể người.
   - `temporal_flip`: Đảo ngược thứ tự thời gian khung hình.
   - `drop_axis`, `drop_joint`: Dropouts ngẫu nhiên để tăng tính bền vững.
   - `gaussian_noise`, `gaussian_filter`: Nhiễu và làm mờ Gaussian.
3. **Biến đổi Động học Khung xương (Kinematic Transforms)**:
   - `joint2bone`: Chuyển đổi tọa độ khớp sang vector xương dựa trên 25 cặp khớp giải phẫu NTU.
   - `bone2joint`: Phục hồi tọa độ khớp từ vector xương và khớp gốc (center joint).
   - `skeleton_adain_bone_length`: Thích ứng độ dài xương (Adaptive Instance Normalization) giữa 2 mẫu xương.
   - `to_motion`: Tính vector vận tốc tức thời $\Delta x_t = x_{t+1} - x_t$.


In [ ]:
# 1. Ghi file feeders/tools.py vào đĩa để tương thích khi DataLoader chạy đa luồng (num_workers > 0)
os.makedirs('feeders', exist_ok=True)
Path('feeders/__init__.py').write_text('', encoding='utf-8')
Path('feeders/tools.py').write_text("import random\nimport numpy as np\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef valid_crop_resize(data_numpy, valid_frame_num, p_interval, window, thres):\n    # input: C,T,V,M\n    if isinstance(p_interval, (int, float)):\n        p_interval = [p_interval]\n    C, T, V, M = data_numpy.shape\n    begin = 0\n    end = valid_frame_num if valid_frame_num > 0 else T\n    valid_size = max(1, end - begin)\n\n    # crop\n    if len(p_interval) == 1:\n        p = p_interval[0]\n        bias = int((1 - p) * valid_size / 2)\n        c_b = min(begin + bias, T - 1)\n        c_e = max(c_b + 1, min(end - bias, T))\n        data = data_numpy[:, c_b:c_e, :, :]  # center_crop\n        cropped_length = data.shape[1]\n    else:\n        p = float(np.random.rand() * (p_interval[1] - p_interval[0]) + p_interval[0])\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)  # constraint cropped_length lower bound as thres\n        max_bias = max(0, valid_size - cropped_length)\n        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0\n        c_b = begin + bias\n        c_e = min(begin + bias + cropped_length, T)\n        data = data_numpy[:, c_b:c_e, :, :]\n        if data.shape[1] == 0:\n            print(cropped_length, bias, valid_size)\n\n    # resize\n    data = torch.tensor(data, dtype=torch.float)  # C, crop_t, V, M\n    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, cropped_length)  # V*M, C, crop_t\n    data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T\n    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()\n    index_t = torch.arange(start=c_b, end=c_e, dtype=torch.float)\n    index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()\n    index_t = 2 * index_t / valid_size - 1\n    return data, index_t.numpy()\n\n\ndef valid_crop_uniform(data_numpy, valid_frame_num, p_interval, window, thres):\n    # input: C,T,V,M\n    if isinstance(p_interval, (int, float)):\n        p_interval = [p_interval]\n    C, T, V, M = data_numpy.shape\n    begin = 0\n    end = valid_frame_num if valid_frame_num > 0 else T\n    valid_size = max(1, end - begin)\n\n    # crop\n    if len(p_interval) == 1:\n        p = p_interval[0]\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)\n        bias = int((1 - p) * valid_size / 2)\n        bias = min(bias, max(0, valid_size - cropped_length))\n\n        if cropped_length < window:\n            inds = np.arange(cropped_length)\n        else:\n            bids = np.array(\n                [i * cropped_length // window for i in range(window + 1)])\n            bst = bids[:window]\n            inds = bst\n\n        inds = inds + bias\n        inds = np.clip(inds, 0, T - 1)\n        data = data_numpy[:, inds, :, :]\n\n    else:\n        p = float(np.random.rand() * (p_interval[1] - p_interval[0]) + p_interval[0])\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres),\n                                    valid_size)  # constraint cropped_length lower bound as 64\n        max_bias = max(0, valid_size - cropped_length)\n        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0\n\n        if cropped_length < window:\n            inds = np.arange(cropped_length)\n        elif window <= cropped_length < 2 * window:\n            basic = np.arange(window)\n            inds = np.random.choice(window + 1, cropped_length - window, replace=False)\n            offset = np.zeros(window + 1, dtype=np.int64)\n            offset[inds] = 1\n            offset = np.cumsum(offset)\n            inds = basic + offset[:-1]\n        else:\n            bids = np.array([i * cropped_length // window for i in range(window + 1)])\n            bsize = np.diff(bids)\n            bst = bids[:window]\n            offset = np.random.randint(bsize)\n            inds = bst + offset\n\n        inds = inds + bias\n        inds = np.clip(inds, 0, T - 1)\n        data = data_numpy[:, inds, :, :]\n        if data.shape[1] == 0:\n            print(cropped_length, bias, valid_size)\n\n    # resize\n    data = torch.tensor(data, dtype=torch.float)\n    index_t = torch.tensor(inds, dtype=torch.float)\n    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, len(inds))  # V*M, C, crop_t\n\n    if len(inds) != window:\n        data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T\n        index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()\n\n    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()\n    index_t = 2 * index_t / valid_size - 1\n    return data, index_t.numpy()\n\n\ndef scale(data_numpy, scale=0.2, p=0.5):\n    if random.random() < p:\n        scale = 1 + np.random.uniform(-1, 1, size=(3, 1, 1, 1)) * np.array(scale)\n        return data_numpy * scale\n    else:\n        return data_numpy.copy()\n\n\n''' AimCLR '''\ntransform_order = {'ntu': [0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7, 16, 17, 18, 19, 12, 13, 14, 15, 20, 23, 24, 21, 22]}\n\n\ndef subtract(data_numpy, p=0.5):\n    joint = random.randint(0, 24)\n\n    C, T, V, M = data_numpy.shape\n    if random.random() < p:\n        data_numpy_new = np.zeros((C, T, V, M))\n        for i in range(V):\n            data_numpy_new[:, :, i, :] = data_numpy[:, :, i, :] - data_numpy[:, :, joint, :]\n        return data_numpy_new\n    else:\n        return data_numpy.copy()\n\n\ndef temporal_flip(data_numpy, index_t, p=0.5):\n    C, T, V, M = data_numpy.shape\n    if random.random() < p:\n        time_range_order = [i for i in range(T)]\n        time_range_reverse = list(reversed(time_range_order))\n        return data_numpy[:, time_range_reverse, :, :], -index_t\n    else:\n        return data_numpy.copy(), index_t.copy()\n\n\ndef spatial_flip(data_numpy, p=0.5):\n    if random.random() < p:\n        index = transform_order['ntu']\n        return data_numpy[:, :, index, :]\n    else:\n        return data_numpy.copy()\n\n\ndef rotate(data_numpy, axis=None, angle=None, p=0.5):\n    if axis != None:\n        axis_next = axis\n    else:\n        axis_next = random.randint(0, 2)\n\n    if angle != None:\n        angle_next = random.uniform(-angle, angle)\n    else:\n        angle_next = random.uniform(-30, 30)\n\n    if random.random() < p:\n        temp = data_numpy.copy()\n        angle = math.radians(angle_next)\n        # x\n        if axis_next == 0:\n            R = np.array([[1, 0, 0],\n                          [0, math.cos(angle), math.sin(angle)],\n                          [0, -math.sin(angle), math.cos(angle)]])\n        # y\n        if axis_next == 1:\n            R = np.array([[math.cos(angle), 0, -math.sin(angle)],\n                          [0, 1, 0],\n                          [math.sin(angle), 0, math.cos(angle)]])\n        # z\n        if axis_next == 2:\n            R = np.array([[math.cos(angle), math.sin(angle), 0],\n                          [-math.sin(angle), math.cos(angle), 0],\n                          [0, 0, 1]])\n        R = R.transpose()\n        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)\n        temp = temp.transpose(3, 0, 1, 2)\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef shear(data_numpy, s1=None, s2=None, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        if s1 != None:\n            s1_list = s1\n        else:\n            s1_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]\n        if s2 != None:\n            s2_list = s2\n        else:\n            s2_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]\n\n        R = np.array([[1, s1_list[0], s2_list[0]],\n                      [s1_list[1], 1, s2_list[1]],\n                      [s1_list[2], s2_list[2], 1]])\n        R = R.transpose()\n        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)\n        temp = temp.transpose(3, 0, 1, 2)\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef drop_axis(data_numpy, axis=None, p=0.5):\n    if axis != None:\n        axis_next = axis\n    else:\n        axis_next = random.randint(0, 2)\n\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n        x_new = np.zeros((T, V, M))\n        temp[axis_next] = x_new\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef drop_joint(data_numpy, joint_list=None, time_range=None, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n\n        if joint_list != None:\n            all_joints = [i for i in range(V)]\n            joint_list_ = random.sample(all_joints, joint_list)\n            joint_list_ = sorted(joint_list_)\n        else:\n            random_int = random.randint(5, 15)\n            all_joints = [i for i in range(V)]\n            joint_list_ = random.sample(all_joints, random_int)\n            joint_list_ = sorted(joint_list_)\n\n        if time_range != None:\n            all_frames = [i for i in range(T)]\n            time_range_ = random.sample(all_frames, time_range)\n            time_range_ = sorted(time_range_)\n        else:\n            random_int = random.randint(16, 32)\n            all_frames = [i for i in range(T)]\n            time_range_ = random.sample(all_frames, random_int)\n            time_range_ = sorted(time_range_)\n\n        x_new = np.zeros((C, len(time_range_), len(joint_list_), M))\n        temp2 = temp[:, time_range_, :, :].copy()\n        temp2[:, :, joint_list_, :] = x_new\n        temp[:, time_range_, :, :] = temp2\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef gaussian_noise(data_numpy, mean=0, std=0.05, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n        noise = np.random.normal(mean, std, size=(C, T, V, M))\n        return temp + noise\n    else:\n        return data_numpy.copy()\n\n\nclass GaussianBlurConv(nn.Module):\n    def __init__(self, channels=3, kernel=15, sigma=[0.1, 2], p=0.5):\n        super(GaussianBlurConv, self).__init__()\n        self.channels = channels\n        self.kernel = kernel\n        self.min_max_sigma = sigma\n        radius = int(kernel / 2)\n        self.kernel_index = np.arange(-radius, radius + 1)\n        self.p = p\n\n    def __call__(self, x):\n        sigma = random.uniform(self.min_max_sigma[0], self.min_max_sigma[1])\n        blur_flter = np.exp(-np.power(self.kernel_index, 2.0) / (2.0 * np.power(sigma, 2.0)))\n        kernel = torch.from_numpy(blur_flter).unsqueeze(0).unsqueeze(0)\n        # kernel =  kernel.float()\n        kernel = kernel.double()\n        kernel = kernel.repeat(self.channels, 1, 1, 1)\n        self.weight = nn.Parameter(data=kernel, requires_grad=False)\n\n        prob = np.random.random_sample()\n        x = torch.from_numpy(x).double()\n        if prob < self.p:\n            x = x.permute(3, 0, 2, 1)\n            x = F.conv2d(x, self.weight, padding=(0, int((self.kernel - 1) / 2)), groups=self.channels)\n            x = x.permute(1, -1, -2, 0)\n\n        return x.numpy()\n\n\ndef gaussian_filter(data_numpy, kernel=15, sig_list=[0.1, 2], p=0.5):\n    g = GaussianBlurConv(3, kernel, sig_list, p)\n    return g(data_numpy)\n\n\n''' Skeleton AdaIN '''\ndef skeleton_adain_bone_length(input, ref): # C T V M\n    eps = 1e-5\n    center = 1\n    ref_c = ref[:, :, center, :]\n\n    # joint to bone (joint2bone)\n    j2b = joint2bone()\n    bone_i = j2b(input) # C T V M\n    bone_r = j2b(ref)\n\n    bone_length_i = np.linalg.norm(bone_i, axis=0) # T V M\n    bone_length_r = np.linalg.norm(bone_r, axis=0)\n\n    bone_length_scale = (bone_length_r + eps) / (bone_length_i + eps) # T V M\n    bone_length_scale = np.expand_dims(bone_length_scale, axis=0) # 1 T V M\n\n    bone_i = bone_i * bone_length_scale\n\n    # bone to joint (bone2joint)\n    b2j = bone2joint()\n    joint = b2j(bone_i, ref_c)\n    return joint\n\n\nclass joint2bone(nn.Module):\n    def __init__(self):\n        super(joint2bone, self).__init__()\n        self.pairs = [(0, 1), (1, 1), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5), (7, 6), (8, 20), (9, 8),\n                      (10, 9), (11, 10), (12, 0), (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),\n                      (19, 18), (20, 1), (21, 7), (22, 7), (23, 11), (24, 11)]\n\n    def __call__(self, joint):\n        bone = np.zeros_like(joint)\n        for v1, v2 in self.pairs:\n            bone[:, :, v1, :] = joint[:, :, v1, :] - joint[:, :, v2, :]\n        return bone\n\n\nclass bone2joint(nn.Module):\n    def __init__(self):\n        super(bone2joint, self).__init__()\n        self.center = 1\n        self.pairs_1 = [(0, 1), (20, 1)]\n        self.pairs_2 = [(2, 20), (4, 20), (8, 20), (12, 0), (16, 0)]\n        self.pairs_3 = [(3, 2), (5, 4), (9, 8), (13, 12), (17, 16)]\n        self.pairs_4 = [(6, 5), (10, 9), (14, 13), (18, 17)]\n        self.pairs_5 = [(7, 6), (11, 10), (15, 14), (19, 18)]\n        self.pairs_6 = [(21, 7), (22, 7), (23, 11), (24, 11)]\n\n    def __call__(self, bone, center):\n        joint = np.zeros_like(bone)\n        joint[:, :, self.center, :] = center\n        for v1, v2 in self.pairs_1:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_2:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_3:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_4:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_5:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_6:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        return joint\n\ndef to_motion(input): # C T V M\n    C, T, V, M = input.shape\n    motion = np.zeros_like(input)\n    motion[:, :T - 1] = np.diff(input, axis=1)\n    return motion", encoding='utf-8')

# 2. Định nghĩa toàn bộ hàm và lớp của tools trực tiếp trong Notebook scope
import random
import numpy as np
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def valid_crop_resize(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        bias = int((1 - p) * valid_size / 2)
        c_b = min(begin + bias, T - 1)
        c_e = max(c_b + 1, min(end - bias, T))
        data = data_numpy[:, c_b:c_e, :, :]  # center_crop
        cropped_length = data.shape[1]
    else:
        p = float(np.random.rand() * (p_interval[1] - p_interval[0]) + p_interval[0])
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)  # constraint cropped_length lower bound as thres
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0
        c_b = begin + bias
        c_e = min(begin + bias + cropped_length, T)
        data = data_numpy[:, c_b:c_e, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)  # C, crop_t, V, M
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, cropped_length)  # V*M, C, crop_t
    data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = torch.arange(start=c_b, end=c_e, dtype=torch.float)
    index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def valid_crop_uniform(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)
        bias = int((1 - p) * valid_size / 2)
        bias = min(bias, max(0, valid_size - cropped_length))

        if cropped_length < window:
            inds = np.arange(cropped_length)
        else:
            bids = np.array(
                [i * cropped_length // window for i in range(window + 1)])
            bst = bids[:window]
            inds = bst

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]

    else:
        p = float(np.random.rand() * (p_interval[1] - p_interval[0]) + p_interval[0])
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres),
                                    valid_size)  # constraint cropped_length lower bound as 64
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0

        if cropped_length < window:
            inds = np.arange(cropped_length)
        elif window <= cropped_length < 2 * window:
            basic = np.arange(window)
            inds = np.random.choice(window + 1, cropped_length - window, replace=False)
            offset = np.zeros(window + 1, dtype=np.int64)
            offset[inds] = 1
            offset = np.cumsum(offset)
            inds = basic + offset[:-1]
        else:
            bids = np.array([i * cropped_length // window for i in range(window + 1)])
            bsize = np.diff(bids)
            bst = bids[:window]
            offset = np.random.randint(bsize)
            inds = bst + offset

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)
    index_t = torch.tensor(inds, dtype=torch.float)
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, len(inds))  # V*M, C, crop_t

    if len(inds) != window:
        data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
        index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()

    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def scale(data_numpy, scale=0.2, p=0.5):
    if random.random() < p:
        scale = 1 + np.random.uniform(-1, 1, size=(3, 1, 1, 1)) * np.array(scale)
        return data_numpy * scale
    else:
        return data_numpy.copy()


''' AimCLR '''
transform_order = {'ntu': [0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7, 16, 17, 18, 19, 12, 13, 14, 15, 20, 23, 24, 21, 22]}


def subtract(data_numpy, p=0.5):
    joint = random.randint(0, 24)

    C, T, V, M = data_numpy.shape
    if random.random() < p:
        data_numpy_new = np.zeros((C, T, V, M))
        for i in range(V):
            data_numpy_new[:, :, i, :] = data_numpy[:, :, i, :] - data_numpy[:, :, joint, :]
        return data_numpy_new
    else:
        return data_numpy.copy()


def temporal_flip(data_numpy, index_t, p=0.5):
    C, T, V, M = data_numpy.shape
    if random.random() < p:
        time_range_order = [i for i in range(T)]
        time_range_reverse = list(reversed(time_range_order))
        return data_numpy[:, time_range_reverse, :, :], -index_t
    else:
        return data_numpy.copy(), index_t.copy()


def spatial_flip(data_numpy, p=0.5):
    if random.random() < p:
        index = transform_order['ntu']
        return data_numpy[:, :, index, :]
    else:
        return data_numpy.copy()


def rotate(data_numpy, axis=None, angle=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if angle != None:
        angle_next = random.uniform(-angle, angle)
    else:
        angle_next = random.uniform(-30, 30)

    if random.random() < p:
        temp = data_numpy.copy()
        angle = math.radians(angle_next)
        # x
        if axis_next == 0:
            R = np.array([[1, 0, 0],
                          [0, math.cos(angle), math.sin(angle)],
                          [0, -math.sin(angle), math.cos(angle)]])
        # y
        if axis_next == 1:
            R = np.array([[math.cos(angle), 0, -math.sin(angle)],
                          [0, 1, 0],
                          [math.sin(angle), 0, math.cos(angle)]])
        # z
        if axis_next == 2:
            R = np.array([[math.cos(angle), math.sin(angle), 0],
                          [-math.sin(angle), math.cos(angle), 0],
                          [0, 0, 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def shear(data_numpy, s1=None, s2=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        if s1 != None:
            s1_list = s1
        else:
            s1_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]
        if s2 != None:
            s2_list = s2
        else:
            s2_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]

        R = np.array([[1, s1_list[0], s2_list[0]],
                      [s1_list[1], 1, s2_list[1]],
                      [s1_list[2], s2_list[2], 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def drop_axis(data_numpy, axis=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        x_new = np.zeros((T, V, M))
        temp[axis_next] = x_new
        return temp
    else:
        return data_numpy.copy()


def drop_joint(data_numpy, joint_list=None, time_range=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape

        if joint_list != None:
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, joint_list)
            joint_list_ = sorted(joint_list_)
        else:
            random_int = random.randint(5, 15)
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, random_int)
            joint_list_ = sorted(joint_list_)

        if time_range != None:
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, time_range)
            time_range_ = sorted(time_range_)
        else:
            random_int = random.randint(16, 32)
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, random_int)
            time_range_ = sorted(time_range_)

        x_new = np.zeros((C, len(time_range_), len(joint_list_), M))
        temp2 = temp[:, time_range_, :, :].copy()
        temp2[:, :, joint_list_, :] = x_new
        temp[:, time_range_, :, :] = temp2
        return temp
    else:
        return data_numpy.copy()


def gaussian_noise(data_numpy, mean=0, std=0.05, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        noise = np.random.normal(mean, std, size=(C, T, V, M))
        return temp + noise
    else:
        return data_numpy.copy()


class GaussianBlurConv(nn.Module):
    def __init__(self, channels=3, kernel=15, sigma=[0.1, 2], p=0.5):
        super(GaussianBlurConv, self).__init__()
        self.channels = channels
        self.kernel = kernel
        self.min_max_sigma = sigma
        radius = int(kernel / 2)
        self.kernel_index = np.arange(-radius, radius + 1)
        self.p = p

    def __call__(self, x):
        sigma = random.uniform(self.min_max_sigma[0], self.min_max_sigma[1])
        blur_flter = np.exp(-np.power(self.kernel_index, 2.0) / (2.0 * np.power(sigma, 2.0)))
        kernel = torch.from_numpy(blur_flter).unsqueeze(0).unsqueeze(0)
        # kernel =  kernel.float()
        kernel = kernel.double()
        kernel = kernel.repeat(self.channels, 1, 1, 1)
        self.weight = nn.Parameter(data=kernel, requires_grad=False)

        prob = np.random.random_sample()
        x = torch.from_numpy(x).double()
        if prob < self.p:
            x = x.permute(3, 0, 2, 1)
            x = F.conv2d(x, self.weight, padding=(0, int((self.kernel - 1) / 2)), groups=self.channels)
            x = x.permute(1, -1, -2, 0)

        return x.numpy()


def gaussian_filter(data_numpy, kernel=15, sig_list=[0.1, 2], p=0.5):
    g = GaussianBlurConv(3, kernel, sig_list, p)
    return g(data_numpy)


''' Skeleton AdaIN '''
def skeleton_adain_bone_length(input, ref): # C T V M
    eps = 1e-5
    center = 1
    ref_c = ref[:, :, center, :]

    # joint to bone (joint2bone)
    j2b = joint2bone()
    bone_i = j2b(input) # C T V M
    bone_r = j2b(ref)

    bone_length_i = np.linalg.norm(bone_i, axis=0) # T V M
    bone_length_r = np.linalg.norm(bone_r, axis=0)

    bone_length_scale = (bone_length_r + eps) / (bone_length_i + eps) # T V M
    bone_length_scale = np.expand_dims(bone_length_scale, axis=0) # 1 T V M

    bone_i = bone_i * bone_length_scale

    # bone to joint (bone2joint)
    b2j = bone2joint()
    joint = b2j(bone_i, ref_c)
    return joint


class joint2bone(nn.Module):
    def __init__(self):
        super(joint2bone, self).__init__()
        self.pairs = [(0, 1), (1, 1), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5), (7, 6), (8, 20), (9, 8),
                      (10, 9), (11, 10), (12, 0), (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),
                      (19, 18), (20, 1), (21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, joint):
        bone = np.zeros_like(joint)
        for v1, v2 in self.pairs:
            bone[:, :, v1, :] = joint[:, :, v1, :] - joint[:, :, v2, :]
        return bone


class bone2joint(nn.Module):
    def __init__(self):
        super(bone2joint, self).__init__()
        self.center = 1
        self.pairs_1 = [(0, 1), (20, 1)]
        self.pairs_2 = [(2, 20), (4, 20), (8, 20), (12, 0), (16, 0)]
        self.pairs_3 = [(3, 2), (5, 4), (9, 8), (13, 12), (17, 16)]
        self.pairs_4 = [(6, 5), (10, 9), (14, 13), (18, 17)]
        self.pairs_5 = [(7, 6), (11, 10), (15, 14), (19, 18)]
        self.pairs_6 = [(21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, bone, center):
        joint = np.zeros_like(bone)
        joint[:, :, self.center, :] = center
        for v1, v2 in self.pairs_1:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_2:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_3:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_4:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_5:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_6:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        return joint

def to_motion(input): # C T V M
    C, T, V, M = input.shape
    motion = np.zeros_like(input)
    motion[:, :T - 1] = np.diff(input, axis=1)
    return motion

# 3. Đăng ký module 'tools' vào sys.modules để cả hai cách gọi:
#    - Gọi trực tiếp: valid_crop_uniform(...)
#    - Gọi qua module: tools.valid_crop_uniform(...)
# đều hoạt động ổn định 100% trong mọi môi trường.
tools_mod = types.ModuleType('tools')
for k, v in list(globals().items()):
    if not k.startswith('_'):
        setattr(tools_mod, k, v)
sys.modules['tools'] = tools_mod
sys.modules['feeders.tools'] = tools_mod

feeders_mod = types.ModuleType('feeders')
setattr(feeders_mod, 'tools', tools_mod)
sys.modules['feeders'] = feeders_mod

print('[Tools] He thong Data Augmentation & Kinematic Tools da san sang!')

# Kiểm tra thử nghiệm nhanh các phép biến đổi
dummy_test_skel = np.random.randn(3, 64, 25, 2).astype(np.float32)
j2b = joint2bone()
bone_test = j2b(dummy_test_skel)
motion_test = to_motion(dummy_test_skel)
assert bone_test.shape == (3, 64, 25, 2), f'Lỗi shape Bone: {bone_test.shape}'
assert motion_test.shape == (3, 64, 25, 2), f'Lỗi shape Motion: {motion_test.shape}'
print(f'[Tools Test] Joint {dummy_test_skel.shape} -> Bone {bone_test.shape} -> Motion {motion_test.shape}')


## 🏋️ Block 4: NTU RGB+D Dataset Feeder (`Feeder` Class)
Lớp `Feeder(Dataset)` kế thừa từ `torch.utils.data.Dataset`, chịu trách nhiệm:
- Đọc file dữ liệu `.npz` nén chứa `x_train`, `y_train`, `x_test`, `y_test`.
- Tự động reshape về cấu trúc chuẩn: `(N, C, T, V, M)` = `(N, 3, 64, 25, 2)`.
- **4 Phương thức dữ liệu (Modalities)**:
  - `data_type='j'`: Joint (tọa độ khớp).
  - `data_type='b'`: Bone (vector xương nối).
  - `data_type='jm'`: Joint-Motion (vận tốc khớp).
  - `data_type='bm'`: Bone-Motion (vận tốc xương).
- **Phân vùng 24 khớp cơ thể (`partition=True`)**:
  - Nhóm 24 khớp thành 6 phân vùng: Tay phải, Tay trái, Chân phải, Chân trái, Thân trên, Thân dưới (loại bỏ khớp số 21 - Spine Base). Cấu trúc này tối ưu cho Part-wise Attention của SkateFormer và Phân cấp L1 của CRL Reasoner.
- **Tích hợp Concept Matrix**: Trả về vector nhãn khái niệm tương ứng $c \in \{0, 1\}^{72}$ cho từng mẫu.


In [ ]:
import numpy as np
import random
import csv

from torch.utils.data import Dataset
import sys, types
if '.' not in sys.path:
    sys.path.insert(0, '.')
try:
    from feeders import tools
except Exception:
    if 'tools' not in globals() or tools is None:
        tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))



class Feeder(Dataset):
    def __init__(self, data_path, label_path=None, p_interval=1, split='train', data_type='j',
                 aug_method='z', intra_p=0.5, inter_p=0.0, window_size=-1,
                 debug=False, thres=64, uniform=False, partition=False,
                 concepts_csv=None, remove_null_concept=True):

        self.debug = debug
        self.data_path = data_path
        self.label_path = label_path
        self.split = split
        self.data_type = data_type
        self.aug_method = aug_method
        self.intra_p = intra_p
        self.inter_p = inter_p
        self.window_size = window_size
        self.p_interval = p_interval
        self.thres = thres
        self.uniform = uniform
        self.partition = partition
        self.concepts_csv = concepts_csv
        self.remove_null_concept = remove_null_concept
        self.load_data()
        self.concept_matrix = None
        self.concept_names = None
        if self.concepts_csv:
            self.load_concepts()
        if partition:
            self.right_arm = np.array([7, 8, 22, 23]) - 1
            self.left_arm = np.array([11, 12, 24, 25]) - 1
            self.right_leg = np.array([13, 14, 15, 16]) - 1
            self.left_leg = np.array([17, 18, 19, 20]) - 1
            self.h_torso = np.array([5, 9, 6, 10]) - 1
            self.w_torso = np.array([2, 3, 1, 4]) - 1
            self.new_idx = np.concatenate((self.right_arm, self.left_arm, self.right_leg, self.left_leg, self.h_torso, self.w_torso), axis=-1)
            # except for joint no.21

    def load_concepts(self):
        """Load REASON concept targets without changing the legacy feeder API."""
        with open(self.concepts_csv, 'r', newline='', encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))

        if not rows or 'action_class' not in rows[0]:
            raise ValueError(f'Invalid concepts CSV: {self.concepts_csv}')

        names = [name for name in rows[0].keys() if name != 'action_class']
        # Nếu CSV có 74 concepts (chưa loại null concepts), lọc bỏ 2 null concepts
        # Nếu CSV đã là 72 concepts chuẩn, giữ nguyên 72 concepts
        if self.remove_null_concept and len(names) == 74:
            names = [name for i, name in enumerate(names) if i not in (50, 65)]
        elif self.remove_null_concept and len(names) > 72:
            names = [name for name in names if name not in ('foot_tap', 'temporal_sequence_seq_cascading')]

        values = []
        for row in rows:
            values.append([float(row[name]) for name in names])
        self.concept_names = names
        self.concept_matrix = np.asarray(values, dtype=np.float32)
        if self.concept_matrix.shape[1] != 72:
            raise ValueError(
                f'Expected 72 concepts after filtering, got {self.concept_matrix.shape[1]}'
            )

    def load_data(self):
        # data: N C V T M
        npz_data = np.load(self.data_path)
        if self.split == 'train':
            self.data = npz_data['x_train']
            self.label = np.where(npz_data['y_train'] > 0)[1]
            self.sample_name = ['train_' + str(i) for i in range(len(self.data))]
        elif self.split == 'test':
            self.data = npz_data['x_test']
            self.label = np.where(npz_data['y_test'] > 0)[1]
            self.sample_name = ['test_' + str(i) for i in range(len(self.data))]
        else:
            raise NotImplementedError('data split only supports train/test')
        N, T, _ = self.data.shape
        self.data = self.data.reshape((N, T, 2, 25, 3)).transpose(0, 4, 1, 3, 2)

    def __len__(self):
        return len(self.label)

    def __iter__(self):
        return self

    def __getitem__(self, index):
        global tools
        if 'tools' not in globals() or tools is None:
            import sys
            try:
                from feeders import tools
            except Exception:
                tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))

        data_numpy = self.data[index]
        label = self.label[index]
        data_numpy = np.array(data_numpy)
        valid_frame_num = np.sum(data_numpy.sum(0).sum(-1).sum(-1) != 0)
        num_people = np.sum(data_numpy.sum(0).sum(0).sum(0) != 0)

        if self.uniform:
            data_numpy, index_t = tools.valid_crop_uniform(data_numpy, valid_frame_num, self.p_interval,
                                                           self.window_size, self.thres)
        else:
            data_numpy, index_t = tools.valid_crop_resize(data_numpy, valid_frame_num, self.p_interval,
                                                          self.window_size, self.thres)

        if self.split == 'train':
            # intra-instance augmentation
            p = np.random.rand()
            if p < self.intra_p:

                if 'a' in self.aug_method:
                    if np.random.rand() < 0.5:
                        data_numpy = data_numpy[:, :, :, np.array([1, 0])]
                if 'b' in self.aug_method:
                    if num_people == 2:
                        if np.random.rand() < 0.5:
                            axis_next = np.random.randint(0, 1)
                            temp = data_numpy.copy()
                            C, T, V, M = data_numpy.shape
                            x_new = np.zeros((C, T, V))
                            temp[:, :, :, axis_next] = x_new
                            data_numpy = temp

                if '1' in self.aug_method:
                    data_numpy = tools.shear(data_numpy, p=0.5)
                if '2' in self.aug_method:
                    data_numpy = tools.rotate(data_numpy, p=0.5)
                if '3' in self.aug_method:
                    data_numpy = tools.scale(data_numpy, p=0.5)
                if '4' in self.aug_method:
                    data_numpy = tools.spatial_flip(data_numpy, p=0.5)
                if '5' in self.aug_method:
                    data_numpy, index_t = tools.temporal_flip(data_numpy, index_t, p=0.5)
                if '6' in self.aug_method:
                    data_numpy = tools.gaussian_noise(data_numpy, p=0.5)
                if '7' in self.aug_method:
                    data_numpy = tools.gaussian_filter(data_numpy, p=0.5)
                if '8' in self.aug_method:
                    data_numpy = tools.drop_axis(data_numpy, p=0.5)
                if '9' in self.aug_method:
                    data_numpy = tools.drop_joint(data_numpy, p=0.5)

            # inter-instance augmentation
            elif (p < (self.intra_p + self.inter_p)) & (p >= self.intra_p):
                adain_idx = random.choice(np.where(self.label == label)[0])
                data_adain = self.data[adain_idx]
                data_adain = np.array(data_adain)
                f_num = np.sum(data_adain.sum(0).sum(-1).sum(-1) != 0)
                t_idx = np.round((index_t + 1) * f_num / 2).astype(np.int64)
                data_adain = data_adain[:, t_idx]
                data_numpy = tools.skeleton_adain_bone_length(data_numpy, data_adain)

            else:
                data_numpy = data_numpy.copy()

        # modality
        if self.data_type == 'b':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
        elif self.data_type == 'jm':
            data_numpy = tools.to_motion(data_numpy)
        elif self.data_type == 'bm':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
            data_numpy = tools.to_motion(data_numpy)
        else:
            data_numpy = data_numpy.copy()

        if self.partition:
            data_numpy = data_numpy[:, :, self.new_idx]

        if self.concept_matrix is not None:
            if label >= len(self.concept_matrix):
                raise IndexError(f'Concept CSV has no row for action label {label}')
            return data_numpy, index_t, label, index, self.concept_matrix[label]
        return data_numpy, index_t, label, index

    def top_k(self, score, top_k):
        rank = score.argsort()
        hit_top_k = [l in rank[i, -top_k:] for i, l in enumerate(self.label)]
        return sum(hit_top_k) * 1.0 / len(hit_top_k)


def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod


print("[Feeder] NTU RGB+D Feeder Class da duoc khoi tao thanh cong!")


## 🤝 Block 5: NTU Interaction Feeder (`FeederInter` Class)
Đối với các bài toán nhận diện hành động tương tác giữa 2 người (Two-person Mutual Actions), lớp `FeederInter` lọc riêng các lớp:
- A49 – A59 (11 hành động: đấm, đá, ôm, bắt tay, chỉ trỏ, v.v.).
- A105 – A119 (15 hành động mở rộng trong NTU 120).
- Tái đánh nhãn về tập chỉ số liên tục $[0 \dots 25]$.
- Giữ nguyên toàn bộ pipeline xử lý 4 modality và phân vùng khớp.


In [ ]:
import numpy as np
import random

from torch.utils.data import Dataset
import sys, types
if '.' not in sys.path:
    sys.path.insert(0, '.')
try:
    from feeders import tools
except Exception:
    if 'tools' not in globals() or tools is None:
        tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))



class FeederInter(Dataset):
    def __init__(self, data_path, label_path=None, p_interval=1, split='train', data_type='j',
                 aug_method='z', intra_p=0.5, inter_p=0.0, window_size=-1,
                 debug=False, thres=64, uniform=False, partition=False):

        self.debug = debug
        self.data_path = data_path
        self.label_path = label_path
        self.split = split
        self.data_type = data_type
        self.aug_method = aug_method
        self.intra_p = intra_p
        self.inter_p = inter_p
        self.window_size = window_size
        self.p_interval = p_interval
        self.thres = thres
        self.uniform = uniform
        self.partition = partition
        self.load_data()
        if partition:
            self.right_arm = np.array([7, 8, 22, 23]) - 1
            self.left_arm = np.array([11, 12, 24, 25]) - 1
            self.right_leg = np.array([13, 14, 15, 16]) - 1
            self.left_leg = np.array([17, 18, 19, 20]) - 1
            self.h_torso = np.array([5, 9, 6, 10]) - 1
            self.w_torso = np.array([2, 3, 1, 4]) - 1
            self.new_idx = np.concatenate((self.right_arm, self.left_arm, self.right_leg, self.left_leg, self.h_torso, self.w_torso), axis=-1)
            # except for joint no.21

    def load_data(self):
        # data: N C V T M
        npz_data = np.load(self.data_path)
        if self.split == 'train':
            data = npz_data['x_train']
            label = np.where(npz_data['y_train'] > 0)[1]
            inter_idx = np.where(((label >= 49) & (label <= 59)) | ((label >= 105) & (label <= 119)))
            self.data = data[inter_idx]
            self.label = label[inter_idx]
            for i in range(len(self.label)):
                if (self.label[i] >= 49) & (self.label[i] <= 59):
                    self.label[i] = self.label[i] - 49
                else:
                    self.label[i] = self.label[i] - 105 + 11
            self.sample_name = ['train_' + str(i) for i in range(len(self.data))]
        elif self.split == 'test':
            data = npz_data['x_test']
            label = np.where(npz_data['y_test'] > 0)[1]
            inter_idx = np.where(((label >= 49) & (label <= 59)) | ((label >= 105) & (label <= 119)))
            self.data = data[inter_idx]
            self.label = label[inter_idx]
            for i in range(len(self.label)):
                if (self.label[i] >= 49) & (self.label[i] <= 59):
                    self.label[i] = self.label[i] - 49
                else:
                    self.label[i] = self.label[i] - 105 + 11
            self.sample_name = ['test_' + str(i) for i in range(len(self.data))]
        else:
            raise NotImplementedError('data split only supports train/test')
        N, T, _ = self.data.shape
        self.data = self.data.reshape((N, T, 2, 25, 3)).transpose(0, 4, 1, 3, 2)

    def __len__(self):
        return len(self.label)

    def __iter__(self):
        return self

    def __getitem__(self, index):
        global tools
        if 'tools' not in globals() or tools is None:
            import sys
            try:
                from feeders import tools
            except Exception:
                tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))

        data_numpy = self.data[index]
        label = self.label[index]
        data_numpy = np.array(data_numpy)
        valid_frame_num = np.sum(data_numpy.sum(0).sum(-1).sum(-1) != 0)
        num_people = np.sum(data_numpy.sum(0).sum(0).sum(0) != 0)

        if self.uniform:
            data_numpy, index_t = tools.valid_crop_uniform(data_numpy, valid_frame_num, self.p_interval,
                                                           self.window_size, self.thres)
        else:
            data_numpy, index_t = tools.valid_crop_resize(data_numpy, valid_frame_num, self.p_interval,
                                                          self.window_size, self.thres)

        if self.split == 'train':
            # intra-instance augmentation
            p = np.random.rand()
            if p < self.intra_p:

                if 'a' in self.aug_method:
                    if np.random.rand() < 0.5:
                        data_numpy = data_numpy[:, :, :, np.array([1, 0])]
                if 'b' in self.aug_method:
                    if num_people == 2:
                        if np.random.rand() < 0.5:
                            axis_next = np.random.randint(0, 1)
                            temp = data_numpy.copy()
                            C, T, V, M = data_numpy.shape
                            x_new = np.zeros((C, T, V))
                            temp[:, :, :, axis_next] = x_new
                            data_numpy = temp

                if '1' in self.aug_method:
                    data_numpy = tools.shear(data_numpy, p=0.5)
                if '2' in self.aug_method:
                    data_numpy = tools.rotate(data_numpy, p=0.5)
                if '3' in self.aug_method:
                    data_numpy = tools.scale(data_numpy, p=0.5)
                if '4' in self.aug_method:
                    data_numpy = tools.spatial_flip(data_numpy, p=0.5)
                if '5' in self.aug_method:
                    data_numpy, index_t = tools.temporal_flip(data_numpy, index_t, p=0.5)
                if '6' in self.aug_method:
                    data_numpy = tools.gaussian_noise(data_numpy, p=0.5)
                if '7' in self.aug_method:
                    data_numpy = tools.gaussian_filter(data_numpy, p=0.5)
                if '8' in self.aug_method:
                    data_numpy = tools.drop_axis(data_numpy, p=0.5)
                if '9' in self.aug_method:
                    data_numpy = tools.drop_joint(data_numpy, p=0.5)

            # inter-instance augmentation
            elif (p < (self.intra_p + self.inter_p)) & (p >= self.intra_p):
                adain_idx = random.choice(np.where(self.label == label)[0])
                data_adain = self.data[adain_idx]
                data_adain = np.array(data_adain)
                f_num = np.sum(data_adain.sum(0).sum(-1).sum(-1) != 0)
                t_idx = np.round((index_t + 1) * f_num / 2).astype(np.int64)
                data_adain = data_adain[:, t_idx]
                data_numpy = tools.skeleton_adain_bone_length(data_numpy, data_adain)

            else:
                data_numpy = data_numpy.copy()

        # modality
        if self.data_type == 'b':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
        elif self.data_type == 'jm':
            data_numpy = tools.to_motion(data_numpy)
        elif self.data_type == 'bm':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
            data_numpy = tools.to_motion(data_numpy)
        else:
            data_numpy = data_numpy.copy()

        if self.partition:
            data_numpy = data_numpy[:, :, self.new_idx]

        return data_numpy, index_t, label, index

    def top_k(self, score, top_k):
        rank = score.argsort()
        hit_top_k = [l in rank[i, -top_k:] for i, l in enumerate(self.label)]
        return sum(hit_top_k) * 1.0 / len(hit_top_k)


def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod



print("[FeederInter] NTU Interaction Feeder Class da duoc khoi tao thanh cong!")


## 🧪 Block 6: Bộ Sinh Dữ liệu Giả lập (Dummy Dataset Generator)
Để đảm bảo notebook có thể **chạy độc lập ngay lập tức** (Zero External Dependency) phục vụ kiểm thử đơn vị (Smoke Testing), hàm dưới đây tự động tạo file `.npz` giả lập định dạng NTU60.


In [ ]:
def create_dummy_ntu_dataset(output_path="data/dummy_ntu60.npz", num_train=8, num_test=4, num_frames=64, num_classes=60):
    """Tạo file dữ liệu giả lập chuẩn định dạng NTU60 (.npz)."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Kích thước chuẩn: N, T, 150 (với 150 = 2 người * 25 khớp * 3 tọa độ)
    x_train = np.random.randn(num_train, num_frames, 150).astype(np.float32)
    # y_train: one-hot vector (N, num_classes)
    y_train = np.zeros((num_train, num_classes), dtype=np.float32)
    for i in range(num_train):
        y_train[i, i % num_classes] = 1.0
        
    x_test = np.random.randn(num_test, num_frames, 150).astype(np.float32)
    y_test = np.zeros((num_test, num_classes), dtype=np.float32)
    for i in range(num_test):
        y_test[i, (i + 1) % num_classes] = 1.0
        
    np.savez_compressed(
        output_path,
        x_train=x_train,
        y_train=y_train,
        x_test=x_test,
        y_test=y_test
    )
    print(f"[Dummy Data] Da tao bo du lieu tai: {output_path}")
    print(f"   - x_train: {x_train.shape}, y_train: {y_train.shape}")
    print(f"   - x_test:  {x_test.shape}, y_test:  {y_test.shape}")
    return output_path

# Tạo file dummy data
dummy_data_path = create_dummy_ntu_dataset()


## 🚀 Block 7: DataLoader Pipeline & Kiểm thử Tự động 4 Modalities
Block này kiểm tra toàn diện pipeline nạp dữ liệu qua cả **4 phương thức**:
- **Joint (`j`)**
- **Bone (`b`)**
- **Joint Motion (`jm`)**
- **Bone Motion (`bm`)**

Kiểm tra xác thực tensor đầu ra:
- Dữ liệu đầu vào: `(Batch_Size, 3, 64, 24, 2)` khi `partition=True`.
- Nhãn hành động: `(Batch_Size,)` kiểu `int64`.
- Nhãn khái niệm (Concept Target): `(Batch_Size, 72)` kiểu `float32` trong khoảng $[0, 1]$.


In [ ]:
def test_feeder_modalities(data_path="data/dummy_ntu60.npz", concept_path="concepts/ntu_rgbd_60_120.csv"):
    modalities = ['j', 'b', 'jm', 'bm']
    modality_names = {
        'j': 'Joint (Khop toa do)',
        'b': 'Bone (Vector xuong)',
        'jm': 'Joint Motion (Van toc khop)',
        'bm': 'Bone Motion (Van toc xuong)'
    }
    
    print("=" * 70)
    print("[TEST] BAT DAU KIEM THU TU DONG FEEDER & DATALOADER TREN 4 MODALITIES")
    print("=" * 70)
    
    for mod in modalities:
        dataset = Feeder(
            data_path=data_path,
            concepts_csv=concept_path,
            split='train',
            data_type=mod,
            window_size=64,
            thres=64,
            uniform=True,
            partition=True,
            remove_null_concept=True,
            p_interval=[0.5, 1.0],
            aug_method='a123489',
            intra_p=0.5,
            inter_p=0.2
        )
        
        loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
        
        # Lấy 1 batch thử nghiệm
        for batch_data, batch_index_t, batch_labels, batch_indices, batch_concepts in loader:
            break
            
        print("")
        print(f"[Modality: {mod.upper()}] - {modality_names[mod]}")
        print(f"   - Skeleton Tensor Shape: {batch_data.shape} (C, T, V, M)")
        print(f"   - Frame Indices Shape:   {batch_index_t.shape}")
        print(f"   - Action Labels Shape:   {batch_labels.shape} (Values: {batch_labels.tolist()})")
        print(f"   - Concept Targets Shape: {batch_concepts.shape} (Expected: [4, 72])")
        
        # Các xác thực nghiêm ngặt
        assert batch_data.shape == (4, 3, 64, 24, 2), f"Lỗi shape dữ liệu {mod}: {batch_data.shape}"
        assert batch_concepts.shape == (4, 72), f"Lỗi shape concept {mod}: {batch_concepts.shape}"
        assert batch_labels.dtype == torch.int64, f"Lỗi kiểu dữ liệu nhãn action: {batch_labels.dtype}"
        assert batch_concepts.dtype == torch.float32, f"Lỗi kiểu dữ liệu concept: {batch_concepts.dtype}"

    print("")
    print("=" * 70)
    print("[PASSED] TAT CA 4 MODALITIES DEU VUOT QUA KIEM THU XAC THUC!")
    print("=" * 70)

# Thực thi kiểm thử
test_feeder_modalities()


## 🎨 Block 8: Trực quan hóa Khung xương & Phân bố Concept
Block này cung cấp các công cụ trực quan hóa trực quan:
1. **Trực quan hóa Khung xương 3D/2D**: Nối 25 khớp cơ thể theo cấu trúc giải phẫu học người (NTU Skeleton Topology).
2. **Biểu đồ Kích hoạt Concept**: Vẽ biểu đồ thanh thể hiện các khái niệm ($1$ trong số $72$ concepts) thuộc về hành động tương ứng.


In [ ]:
# Danh sách các cặp liên kết xương giải phẫu NTU 25 khớp
NTU_BONES = [
    (0, 1), (1, 20), (20, 2), (2, 3),          # Thân giữa & Cổ - Đầu
    (20, 4), (4, 5), (5, 6), (6, 7), (7, 21), (7, 22),   # Tay trái
    (20, 8), (8, 9), (9, 10), (10, 11), (11, 23), (11, 24), # Tay phải
    (0, 12), (12, 13), (13, 14), (14, 15),     # Chân trái
    (0, 16), (16, 17), (17, 18), (18, 19)      # Chân phải
]

def visualize_skeleton_and_concepts(data_numpy, action_idx, ontology, frame_idx=0, person_idx=0):
    """Trực quan hóa khung xương 2D và biểu đồ concepts được kích hoạt."""
    fig = plt.figure(figsize=(14, 5))
    
    # 1. Vẽ Khung xương 2D
    ax1 = fig.add_subplot(1, 2, 1)
    C, T, V, M = data_numpy.shape
    x = data_numpy[0, frame_idx, :, person_idx]
    y = data_numpy[1, frame_idx, :, person_idx]
    
    ax1.scatter(x, y, c='red', s=40, zorder=3, label='Khớp (Joints)')
    for i, (xi, yi) in enumerate(zip(x, y)):
        ax1.text(xi + 0.01, yi + 0.01, str(i), fontsize=7, color='darkblue')
        
    # Nối các xương nếu chỉ số khớp nằm trong phạm vi
    for v1, v2 in NTU_BONES:
        if v1 < V and v2 < V:
            ax1.plot([x[v1], x[v2]], [y[v1], y[v2]], 'b-', alpha=0.6, linewidth=2)
            
    ax1.set_title(f"Khung xuong (T={frame_idx}, Person={person_idx})", fontsize=12, fontweight='bold')
    ax1.set_xlabel("Truc X")
    ax1.set_ylabel("Truc Y")
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend()
    
    # 2. Vẽ Biểu đồ Active Concepts
    ax2 = fig.add_subplot(1, 2, 2)
    active_concepts = ontology.get_active_concepts(action_idx)
    action_name = ontology.action_names[action_idx] if action_idx < len(ontology.action_names) else f"Action_{action_idx}"
    
    if len(active_concepts) == 0:
        ax2.text(0.5, 0.5, "Khong co Concept nao duoc kich hoat", ha='center', va='center')
    else:
        y_pos = np.arange(len(active_concepts))
        ax2.barh(y_pos, [1.0] * len(active_concepts), color='teal', alpha=0.8)
        ax2.set_yticks(y_pos)
        ax2.set_yticklabels(active_concepts, fontsize=9)
        ax2.invert_yaxis()
        ax2.set_xlim(0, 1.2)
        ax2.set_xlabel("Gia tri Kich hoat (Ground Truth Concept Target)")
        
    ax2.set_title(f"Active Concepts ({len(active_concepts)}/72): {action_name}", fontsize=12, fontweight='bold')
    ax2.grid(axis='x', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# Trực quan hóa thử nghiệm một mẫu
vis_dataset = Feeder(
    data_path="data/dummy_ntu60.npz",
    concepts_csv="concepts/ntu_rgbd_60_120.csv",
    split='train',
    data_type='j',
    window_size=64,
    thres=64,
    uniform=True,
    partition=False  # Giữ nguyên 25 khớp để nối đúng toàn bộ xương
)
sample_data, _, sample_label, _, _ = vis_dataset[0]
print(f"[Visualizer] Hien thi truc quan cho mau 0 (Action: {sample_label}):")
visualize_skeleton_and_concepts(sample_data, sample_label, ontology, frame_idx=0, person_idx=0)


---

## 📌 PHẦN II, III, IV, V: LỘ TRÌNH MỞ RỘNG TIẾP THEO

File Notebook này được thiết kế theo cấu trúc module đồng nhất để mở rộng tiếp tục cho các thành phần tiếp theo của CLIR:
1. **Phần II: Perception Backbone (SkateFormer Architecture)**:
   - `SkateFormer` trích xuất đặc trưng không gian - thời gian theo từng phân vùng cơ thể (Part-wise Spatial Attention & Temporal Attention).
2. **Phần III: Tầng Logic CRL (Concept Reasoner with Logic)**:
   - `LiteralLayer`: Chuyển đổi xác suất concept thành literal logic với Straight-Through Estimator (`BinarizeSTE`).
   - `ConjunctionLayer`: Hội logic L1 (part-grouped) và L2a (AND2).
   - `DisjunctionLayer`: Tuyển logic L2b (OR1).
   - `CRLReasoner`: Tính toán bằng chứng logic (Logic Evidence $r^t \in \mathbb{R}^{512}$).
3. **Phần IV: Vòng lặp Tinh chỉnh Khái niệm (ConceptUpdateCell & CLIRSkateFormer)**:
   - Tinh chỉnh khái niệm vòng lặp $K=2$ qua `ConceptUpdateCell` và phân lớp hành động nghiêm ngặt bằng `ActionHead`.
4. **Phần V: Huấn luyện Pure STE Annealing & Trích xuất Luật DNF (RuleExtractor)**:
   - Vòng lặp huấn luyện tối ưu hóa với Temperature Annealing $2.0 \rightarrow 10.0$.
   - Trích xuất tự động luật DNF cho từng hành động kèm đánh giá Fidelity, Precision, Coverage.
